# 05 Hidden Layer Optimization

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb008_s1_width.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_width.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s2_depth.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_depth.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s1_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s2_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_width.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'd1662f644284c3c674ed774b8d2bc8ee22c4a5c4'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_depth.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'ce9d01508df0c366f12c39a885425b2b0f93ca5d'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '17381023fe5802b73523b1e299765d78c433dca1'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '5a012e52c8d9dfbce178635262c52301f5882f5b'}}

## exp_cmb008_s1_width

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_width.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_width.py)  
Git blob: `d1662f644284c3c674ed774b8d2bc8ee22c4a5c4`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 / S1. Fixed 30/28 + CoM; vary only single hidden width.

S1 widths 64, 128, 256 use unchanged EXP-CMB-005 data/augmentation functions;
baseline width 100 is reused from the archived runs.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.

CLI 인자 (정적 추출): `--arm`, `--width`, `--seed`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_width.py
#!/usr/bin/env python3
"""EXP-CMB-008 / S1. Fixed 30/28 + CoM; vary only single hidden width.

S1 widths 64, 128, 256 use unchanged EXP-CMB-005 data/augmentation functions;
baseline width 100 is reused from the archived runs.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,sys,time,warnings
from pathlib import Path
import joblib
import numpy as np
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,roc_auc_score,
                             precision_recall_fscore_support,confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT=Path(__file__).resolve().parents[4]
CMB=ROOT/"05_model_optimize"/"MLP"/"Combined"
NORMAL=ROOT/"05_model_optimize"/"MLP"/"Normal"
sys.path.insert(0,str(CMB/"scripts"))
sys.path.insert(0,str(NORMAL/"scripts"))
from combined_mlp_writer_cv import (load_official,load_writers,PixelScaler255,
                                    WRITERS,LABELS,V3_SHA,HANDMADE_SHA)
from exp006_size_com_comparison import process,image_geometry
from exp015_scale_stroke_grid import nested_subset
from exp016_high_fraction_grid import scale_base,stroke_jitter
from exp014_balanced_augmentation import balanced_indices
from exp011_012_causal_probes import fixed_scale

SCALES=(.8,.9,1.,1.1,1.2)
PAIRS=(("0","D"),("D","0"),("8","B"),("B","8"),("5","F"),("F","5"),
       ("7","9"),("7","A"),("4","F"))
ARMS={"com_only":(0.,0.),
      "scale10_stroke00":(.1,0.),
      "scale10_stroke14":(.1,.14),
      "scale10_stroke28":(.1,.28),
      "scale20_stroke00":(.2,0.),
      "scale20_stroke14":(.2,.14),
      "scale20_stroke28":(.2,.28),
      "scale30_stroke00":(.3,0.),
      "scale30_stroke14":(.3,.14),
      "scale30_stroke28":(.3,.28)}
TARGETS=(16,18,20,22,24)
SEEDS=(42,43,44)
CLASSES=np.arange(16,dtype=int)

def savecsv(p,records):
    records=list(records)
    if not records:raise RuntimeError("missing rows "+str(p))
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        wr=csv.DictWriter(f,fieldnames=list(records[0]))
        wr.writeheader();wr.writerows(records)

def metrics(y,p,subset=None):
    labels=CLASSES if subset is None else np.asarray(subset)
    prec,rec,f1,sup=precision_recall_fscore_support(y,p,labels=labels,zero_division=0)
    return {"n":int(len(y)),"correct":int(np.sum(p==y)),
            "accuracy":float(np.mean(p==y)),
            "macro_f1":float(f1.mean()),
            "macro_precision":float(prec.mean()),
            "macro_recall":float(rec.mean())}

def confusion_pairs(y,p,writer,arm,seed,source):
    """All 240 off-diagonal directions, including zero-error pairs.

    Denominator n_actual corresponds to the *actual* (row) class.
    Both counts and conditional percentages are saved; total overall
    accuracy does not replace per-class and per-pair error analysis.
    """
    cm=confusion_matrix(y,p,labels=CLASSES)
    result=[]
    nall=int(len(y))
    nwrong=int(nall-np.trace(cm))
    for actual in CLASSES:
        nactual=int(cm[actual].sum())
        wrong=int(nactual-cm[actual,actual])
        for target in CLASSES:
            if actual==target:continue
            ct=int(cm[actual,target])
            result.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
                 "actual":LABELS[actual],"predicted":LABELS[target],
                 "n_actual":nactual,"n_actual_wrong":wrong,
                 "n_dataset":nall,"n_dataset_wrong":nwrong,
                 "errors":ct,"error_rate_within_actual_class":ct/nactual if nactual else 0,
                 "share_of_actual_class_errors":ct/wrong if wrong else 0,
                 "share_of_all_dataset_errors":ct/nwrong if nwrong else 0})
    assert len(result)==16*15
    assert sum(int(r["errors"]) for r in result)==nwrong
    return result

def total_errors_by_class(y,p,writer,arm,seed,source):
    """All 16 true classes: total incorrect / n actual class, plus group switch."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    out=[]
    for a in CLASSES:
        n=int(cm[a].sum());correct=int(cm[a,a]);wrong=n-correct
        family_swap=int(cm[a,10:].sum()) if a<10 else int(cm[a,:10].sum())
        out.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
           "actual":LABELS[a],"n":n,"correct":correct,"wrong":wrong,
           "class_error_rate":wrong/n if n else 0,
           "opposite_family_errors":family_swap,
           "opposite_family_error_rate":family_swap/n if n else 0})
    assert len(out)==16
    return out

def family_error_breakdown(y,p,writer,arm,seed,source):
    """Digits→letters and letters→digits alongside within-family mistakes."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    d2l=int(cm[:10,10:].sum());l2d=int(cm[10:,:10].sum())
    d2d=int(cm[:10,:10].sum()-np.trace(cm[:10,:10]))
    l2l=int(cm[10:,10:].sum()-np.trace(cm[10:,10:]))
    return [{"arm":arm,"seed":seed,"dataset":source,"writer":writer,
      "direction":key,"errors":val,"n_actual_family":n,
      "error_rate":val/n if n else 0}
      for key,val,n in (("digit_to_letter",d2l,int(cm[:10].sum())),
                        ("letter_to_digit",l2d,int(cm[10:].sum())),
                        ("digit_to_digit",d2d,int(cm[:10].sum())),
                        ("letter_to_letter",l2l,int(cm[10:].sum())))]

def geom_rows(name,x,y,arm,seed):
    geo=image_geometry(x)
    out=[]
    for nameg,mask in (("digits",y<10),("letters",y>=10)):
        z={"arm":arm,"seed":seed,"dataset":name,"group":nameg,
           "n":int(mask.sum())}
        for field in ("long_side","width","height","mass"):
            zz=np.asarray(geo[field])[mask]
            z[field+"_mean"]=float(zz.mean())
            z[field+"_std"]=float(zz.std())
        out.append(z)
    return out

def binary_size_probe(xtrain,ytrain,xval,yval):
    train=image_geometry(xtrain)["long_side"].reshape(-1,1)
    val=image_geometry(xval)["long_side"].reshape(-1,1)
    model=Pipeline([("std",StandardScaler()),
      ("logistic",LogisticRegression(class_weight="balanced",random_state=42,max_iter=1000))])
    model.fit(train,(ytrain>=10).astype(int))
    q=model.predict_proba(val)[:,1]
    return {"bal_acc":float(balanced_accuracy_score(yval>=10,q>=.5)),
            "auc":float(roc_auc_score(yval>=10,q))}

def prepareraw(official,writers,arm):
    ratio,stroke=ARMS[arm]
    raw=official["X_train"]
    yy=official["y_train"]
    # CoM-only is exactly the EXP-CMB-004 reference condition.
    main=process(raw,"com",20)
    changed={"method":arm,"requested_ratio":ratio,"actual_ratio":0.,
             "n_aug":0,"n_stroke_changed":0,"n_stroke_rejected":0,
             "official_train_rows":len(raw)}
    targets=[]
    if ratio:
        ids,target_map=nested_subset(yy,ratio)  # Normal EXP015 nested low-ratio group
        half,_=balanced_indices(yy,seed=20261116)
        augmented=scale_base(raw,ids,target_map)
        augmented,stroke_stats=stroke_jitter(augmented,stroke,ids,half)
        main[ids]=process(augmented,"com",20)
        changed.update({"n_aug":len(ids),"actual_ratio":len(ids)/len(yy),
             "n_stroke_changed":stroke_stats["modified"],
             "n_stroke_rejected":stroke_stats["rejected"]})
        for cl in CLASSES:
            for length in TARGETS:
                ct=int(np.sum((yy[ids]==cl)&(target_map[ids]==length)))
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":ct})
                assert ct>=1
        for cl in CLASSES:
            cc=[x["train_images_assigned"] for x in targets if x["class"]==LABELS[cl]]
            assert max(cc)-min(cc)<=1
        assert abs(len(ids)/len(yy)-ratio)<.002
    else:
        for cl in CLASSES:
            for length in TARGETS:
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":0})
    transformed_w={w:process(writers[w]["X"],"com",20) for w in WRITERS}
    assert main.shape==(76800,784) and len(transformed_w)==4
    return main,transformed_w,changed,targets

def make_scaled(official,writers):
    ref={}
    for scale in SCALES:
        # exact Normal EXP012 intervention; raw image BBox-centered affine, then CoM
        ref[("official",scale)]=process(fixed_scale(official["X_val"],scale),"com",20)
        for w in WRITERS:
            ref[(w,scale)]=process(fixed_scale(writers[w]["X"],scale),"com",20)
    return ref

def eval_scales(model,raw_map,y,dataset,writer,arm,seed):
    rows=[]; pair_rows=[]; focus=[]; class_errors=[]; family_errors=[]
    probs={}
    for sc in SCALES:
        xx=raw_map[(writer if dataset=="OOF" else "official",sc)]
        q=np.asarray(model.predict_proba(xx),dtype=np.float64)
        assert np.array_equal(model.named_steps["model"].classes_,CLASSES)
        p=q.argmax(axis=1).astype(int)
        letter=q[:,10:].sum(axis=1)
        probs[sc]=(p,letter)
    _,base_letter=probs[1.]
    base_family=base_letter>=.5
    for sc in SCALES:
        p,letter=probs[sc]
        for group,z,cls in (("all",np.ones(len(y),dtype=bool),CLASSES),
             ("digits",y<10,np.arange(10)),("letters",y>=10,np.arange(10,16))):
            m=metrics(y[z],p[z],cls)
            flip=(letter[z]>=.5)!=(base_family[z])
            rows.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
               "scale":sc,"group":group,**m,
               "family_flip":float(flip.mean()),
               "mean_abs_delta_p_letter":float(np.abs(letter[z]-base_letter[z]).mean()),
               "mean_p_letter":float(letter[z].mean())})
        for cls_i in CLASSES:
            z=y==cls_i
            pz=p[z];ll=letter[z];bb=base_letter[z]
            focus.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
                "scale":sc,"actual":LABELS[cls_i],"n":int(z.sum()),
                "correct":int(np.sum(pz==cls_i)),
                "accuracy":float(np.mean(pz==cls_i)),
                "family_flip":float(np.mean((ll>=.5)!=(bb>=.5))),
                "mean_abs_delta_p_letter":float(np.mean(np.abs(ll-bb))),
                "mean_p_letter":float(ll.mean())})
        sc_name=dataset+"_scale_"+str(sc)
        pair_rows+=confusion_pairs(y,p,writer,arm,seed,sc_name)
        class_errors+=total_errors_by_class(y,p,writer,arm,seed,sc_name)
        family_errors+=family_error_breakdown(y,p,writer,arm,seed,sc_name)
    return rows,pair_rows,focus,probs[1.][0],class_errors,family_errors

def output_plots(y,p,out,arm,seed):
    cm=confusion_matrix(y,p,labels=CLASSES)
    for k in ("counts","row_normalized"):
        v=cm if k=="counts" else 100*cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
        fig,ax=plt.subplots(figsize=(10,8))
        im=ax.imshow(v,cmap="viridis",vmin=0,vmax=160 if k=="counts" else 100)
        fig.colorbar(im,ax=ax,fraction=.045,pad=.04)
        ax.set(xticks=np.arange(16),yticks=np.arange(16),
               xticklabels=list(LABELS),yticklabels=list(LABELS),
               xlabel="Predicted",ylabel="True",
               title=f"EXP-CMB-005 {arm} seed={seed} Combined OOF {k}")
        fig.tight_layout()
        fig.savefig(out/f"OOF_confusion_{k}.png",dpi=145)
        plt.close(fig)

def run(arm,seed,out,width):
    if arm!="scale30_stroke28" or seed not in SEEDS or width not in (64,128,256):raise ValueError((arm,seed,width))
    if out.exists(): raise FileExistsError(out)
    out.mkdir(parents=True)
    (out/"models").mkdir()
    t0=time.perf_counter()
    original=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
    handmade=ROOT/"02_data"/"Hand_made"
    off=load_official(original)
    writers=load_writers(handmade)
    assert off["sha256"]==V3_SHA
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    assert len(off["X_train"])==76800 and len(off["X_val"])==19200
    tpre=time.perf_counter()
    xtrain_off,xwriters,info,targets=prepareraw(off,writers,arm)
    xval=process(off["X_val"],"com",20)
    scale_map=make_scaled(off,writers)
    preprocess_time=time.perf_counter()-tpre
    official_probe=binary_size_probe(xtrain_off,off["y_train"],xval,off["y_val"])
    savecsv(out/"target_assignments.csv",targets)
    features=geom_rows("official_train",xtrain_off,off["y_train"],arm,seed)
    features+=geom_rows("official_val",xval,off["y_val"],arm,seed)
    for w in WRITERS:features+=geom_rows("writer_"+str(w),xwriters[w],writers[w]["y"],arm,seed)
    savecsv(out/"geometry_distributions.csv",features)
    foldrows=[];pairs=[];class_errors=[];family_errors=[];scale_rows=[];class_rows=[];scale_class_rows=[];pred_rows=[];loss=[];fit=[]
    pooled=[]
    for held in WRITERS:
        trwriters=[w for w in WRITERS if w!=held]
        X=np.concatenate([xtrain_off]+[xwriters[w] for w in trwriters])
        y=np.concatenate([off["y_train"]]+[writers[w]["y"] for w in trwriters])
        assert X.shape==(78720,784) and len(y)==78720
        mlp=Pipeline([("scale_0_1",PixelScaler255()),
                      ("model",MLPClassifier(random_state=seed,hidden_layer_sizes=(width,)))])
        started=time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            mlp.fit(X,y)
        sec=time.perf_counter()-started
        mlpnet=mlp.named_steps["model"]
        assert mlpnet.n_features_in_==784
        for it,lossval in enumerate(mlpnet.loss_curve_):
            loss.append({"arm":arm,"seed":seed,"heldout":held,
                        "iteration":it+1,"loss":float(lossval)})
        fit.append({"arm":arm,"seed":seed,"heldout":held,
                    "train_n":len(y),"heldout_n":640,
                    "fit_seconds":sec,"iterations":mlpnet.n_iter_,
                    "last_loss":float(mlpnet.loss_),
                    "warnings":" | ".join(str(w.message) for w in caught)})
        fout=out/"models"/f"heldout_{held}.joblib"
        joblib.dump(mlp,fout,compress=3)
        handwritten_y=writers[held]["y"]
        outcome=mlp.predict(xwriters[held]).astype(int)
        pp=mlp.predict(xval).astype(int)
        for ds,ytrue,pred in (("OOF",handwritten_y,outcome),
                              ("OfficialVal",off["y_val"],pp)):
            maskset=(("all",np.ones(len(ytrue),bool),CLASSES),
                     ("digits",ytrue<10,np.arange(10)),
                     ("letters",ytrue>=10,np.arange(10,16)))
            for g,z,classes in maskset:
                foldrows.append({"arm":arm,"seed":seed,"heldout":held,"dataset":ds,
                      "group":g,**metrics(ytrue[z],pred[z],classes)})
        prec,rec,f1,_=precision_recall_fscore_support(handwritten_y,outcome,
              labels=CLASSES,zero_division=0)
        for ci in CLASSES:
            z=handwritten_y==ci
            class_rows.append({"arm":arm,"seed":seed,"writer":held,
               "class":LABELS[ci],"n":int(z.sum()),"correct":int(np.sum(outcome[z]==ci)),
               "precision":float(prec[ci]),"recall":float(rec[ci]),"f1":float(f1[ci])})
        for i in range(640):
            pred_rows.append({"arm":arm,"seed":seed,"writer":held,"index":i,
                "actual":LABELS[int(handwritten_y[i])],
                "pred":LABELS[int(outcome[i])]})
        pooled.append(outcome)
        pairs+=confusion_pairs(handwritten_y,outcome,held,arm,seed,"OOF_original")
        pairs+=confusion_pairs(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        class_errors+=total_errors_by_class(handwritten_y,outcome,held,arm,seed,"OOF_original")
        class_errors+=total_errors_by_class(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        family_errors+=family_error_breakdown(handwritten_y,outcome,held,arm,seed,"OOF_original")
        family_errors+=family_error_breakdown(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        for ds,ytrue,wid in (("OOF",handwritten_y,held),
                             ("OfficialVal",off["y_val"],held)):
            z,p2,c,base,ce,fe=eval_scales(mlp,scale_map,ytrue,ds,wid,arm,seed)
            scale_rows+=z;pairs+=p2;scale_class_rows+=c
            class_errors+=ce;family_errors+=fe
            if ds=="OOF":assert np.array_equal(base,outcome)
            else:assert np.array_equal(base,pp)
        print("FOLD_COMPLETED",arm,seed,held,
              "OOF",round(np.mean(outcome==handwritten_y),6),
              "VAL",round(np.mean(pp==off["y_val"]),6),
              "fit_s",round(sec,2),flush=True)
        del X,y,mlp
    hand_y=np.concatenate([writers[w]["y"] for w in WRITERS])
    pooled_p=np.concatenate(pooled)
    assert len(hand_y)==2560
    output_plots(hand_y,pooled_p,out,arm,seed)
    savecsv(out/"fold_metrics.csv",foldrows)
    savecsv(out/"oof_per_class.csv",class_rows)
    savecsv(out/"per_class_and_scale.csv",scale_class_rows)
    savecsv(out/"pair_confusions.csv",pairs)
    savecsv(out/"per_class_error_rates.csv",class_errors)
    savecsv(out/"family_confusion_rates.csv",family_errors)
    savecsv(out/"scale_sensitivity.csv",scale_rows)
    savecsv(out/"oof_predictions.csv",pred_rows)
    savecsv(out/"training_loss.csv",loss)
    savecsv(out/"training_time.csv",fit)
    overall=metrics(hand_y,pooled_p)
    meta={"experiment":"EXP-CMB-008","stage":"S1_width","width":width,"status":"S1 model trained, selection pending",
          "arm":arm,"seed":seed,"handmade_cv":"Leave-One-Writer-Out 4-Fold",
          "official_n":76800,"official_val_n":19200,"training_handmade_each_fold":1920,
          "oof_n":2560,"scale_interventions":SCALES,"class_balanced_targets":TARGETS,
          "official_train_only_augmentation":True,
          "handmade_train_preprocess":"CoM-only",
          "heldout_and_official_val_preprocess":"CoM-only",
          "augmentation_seed":20261116,"augmentation":info,
          "preregistration":"EXP-CMB-008 S1; 30% Size 28% Stroke, CoM fixed; widths 64/128/256; no test",
          "hyperparameter_search":True,"only_changed_parameter":"hidden_layer_sizes",
          "fixed_params":{"activation":"relu","solver":"adam","alpha":0.0001,
             "learning_rate_init":0.001,"max_iter":200,"early_stopping":False},
          "official_sha256":off["sha256"],
          "handmade_sha256":{str(w):writers[w]["sha256"] for w in WRITERS},
          "official_train_size_binary_probe":official_probe,
          "oof_metrics":overall,"fit_total_seconds":sum(r["fit_seconds"] for r in fit),
          "preprocessing_seconds":preprocess_time,
          "wall_seconds":time.perf_counter()-t0,
          "original_test_used":False,"professor_test_used":False,
          "new_handmade_480_used":False,
          "versions":{"numpy":np.__version__,"scikit_learn":sklearn.__version__},
          "model_sha256":{str(w):hashlib.sha256((out/"models"/f"heldout_{w}.joblib").read_bytes()).hexdigest()
                          for w in WRITERS}}
    (out/"manifest.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("COMPLETED",arm,seed,json.dumps(overall),flush=True)

if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--arm",choices=("scale30_stroke28",),default="scale30_stroke28")
    ap.add_argument("--width",type=int,choices=(64,128,256),required=True)
    ap.add_argument("--seed",type=int,choices=SEEDS,required=True)
    ap.add_argument("--out",type=Path,required=True)
    ar=ap.parse_args()
    run(ar.arm,ar.seed,ar.out,ar.width)


## exp_cmb008_s2_depth

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_depth.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_depth.py)  
Git blob: `ce9d01508df0c366f12c39a885425b2b0f93ca5d`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 / S2. Freeze first hidden width 256, vary second hidden layer.

S2 depths 128, 256 add one hidden layer to 256-wide S1 winner, while keeping
EXP-CMB-005 input augmentation / CoM and all other HP frozen.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.

CLI 인자 (정적 추출): `--arm`, `--depth`, `--seed`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_depth.py
#!/usr/bin/env python3
"""EXP-CMB-008 / S2. Freeze first hidden width 256, vary second hidden layer.

S2 depths 128, 256 add one hidden layer to 256-wide S1 winner, while keeping
EXP-CMB-005 input augmentation / CoM and all other HP frozen.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,sys,time,warnings
from pathlib import Path
import joblib
import numpy as np
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,roc_auc_score,
                             precision_recall_fscore_support,confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT=Path(__file__).resolve().parents[4]
CMB=ROOT/"05_model_optimize"/"MLP"/"Combined"
NORMAL=ROOT/"05_model_optimize"/"MLP"/"Normal"
sys.path.insert(0,str(CMB/"scripts"))
sys.path.insert(0,str(NORMAL/"scripts"))
from combined_mlp_writer_cv import (load_official,load_writers,PixelScaler255,
                                    WRITERS,LABELS,V3_SHA,HANDMADE_SHA)
from exp006_size_com_comparison import process,image_geometry
from exp015_scale_stroke_grid import nested_subset
from exp016_high_fraction_grid import scale_base,stroke_jitter
from exp014_balanced_augmentation import balanced_indices
from exp011_012_causal_probes import fixed_scale

SCALES=(.8,.9,1.,1.1,1.2)
PAIRS=(("0","D"),("D","0"),("8","B"),("B","8"),("5","F"),("F","5"),
       ("7","9"),("7","A"),("4","F"))
ARMS={"com_only":(0.,0.),
      "scale10_stroke00":(.1,0.),
      "scale10_stroke14":(.1,.14),
      "scale10_stroke28":(.1,.28),
      "scale20_stroke00":(.2,0.),
      "scale20_stroke14":(.2,.14),
      "scale20_stroke28":(.2,.28),
      "scale30_stroke00":(.3,0.),
      "scale30_stroke14":(.3,.14),
      "scale30_stroke28":(.3,.28)}
TARGETS=(16,18,20,22,24)
SEEDS=(42,43,44)
CLASSES=np.arange(16,dtype=int)

def savecsv(p,records):
    records=list(records)
    if not records:raise RuntimeError("missing rows "+str(p))
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        wr=csv.DictWriter(f,fieldnames=list(records[0]))
        wr.writeheader();wr.writerows(records)

def metrics(y,p,subset=None):
    labels=CLASSES if subset is None else np.asarray(subset)
    prec,rec,f1,sup=precision_recall_fscore_support(y,p,labels=labels,zero_division=0)
    return {"n":int(len(y)),"correct":int(np.sum(p==y)),
            "accuracy":float(np.mean(p==y)),
            "macro_f1":float(f1.mean()),
            "macro_precision":float(prec.mean()),
            "macro_recall":float(rec.mean())}

def confusion_pairs(y,p,writer,arm,seed,source):
    """All 240 off-diagonal directions, including zero-error pairs.

    Denominator n_actual corresponds to the *actual* (row) class.
    Both counts and conditional percentages are saved; total overall
    accuracy does not replace per-class and per-pair error analysis.
    """
    cm=confusion_matrix(y,p,labels=CLASSES)
    result=[]
    nall=int(len(y))
    nwrong=int(nall-np.trace(cm))
    for actual in CLASSES:
        nactual=int(cm[actual].sum())
        wrong=int(nactual-cm[actual,actual])
        for target in CLASSES:
            if actual==target:continue
            ct=int(cm[actual,target])
            result.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
                 "actual":LABELS[actual],"predicted":LABELS[target],
                 "n_actual":nactual,"n_actual_wrong":wrong,
                 "n_dataset":nall,"n_dataset_wrong":nwrong,
                 "errors":ct,"error_rate_within_actual_class":ct/nactual if nactual else 0,
                 "share_of_actual_class_errors":ct/wrong if wrong else 0,
                 "share_of_all_dataset_errors":ct/nwrong if nwrong else 0})
    assert len(result)==16*15
    assert sum(int(r["errors"]) for r in result)==nwrong
    return result

def total_errors_by_class(y,p,writer,arm,seed,source):
    """All 16 true classes: total incorrect / n actual class, plus group switch."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    out=[]
    for a in CLASSES:
        n=int(cm[a].sum());correct=int(cm[a,a]);wrong=n-correct
        family_swap=int(cm[a,10:].sum()) if a<10 else int(cm[a,:10].sum())
        out.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
           "actual":LABELS[a],"n":n,"correct":correct,"wrong":wrong,
           "class_error_rate":wrong/n if n else 0,
           "opposite_family_errors":family_swap,
           "opposite_family_error_rate":family_swap/n if n else 0})
    assert len(out)==16
    return out

def family_error_breakdown(y,p,writer,arm,seed,source):
    """Digits→letters and letters→digits alongside within-family mistakes."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    d2l=int(cm[:10,10:].sum());l2d=int(cm[10:,:10].sum())
    d2d=int(cm[:10,:10].sum()-np.trace(cm[:10,:10]))
    l2l=int(cm[10:,10:].sum()-np.trace(cm[10:,10:]))
    return [{"arm":arm,"seed":seed,"dataset":source,"writer":writer,
      "direction":key,"errors":val,"n_actual_family":n,
      "error_rate":val/n if n else 0}
      for key,val,n in (("digit_to_letter",d2l,int(cm[:10].sum())),
                        ("letter_to_digit",l2d,int(cm[10:].sum())),
                        ("digit_to_digit",d2d,int(cm[:10].sum())),
                        ("letter_to_letter",l2l,int(cm[10:].sum())))]

def geom_rows(name,x,y,arm,seed):
    geo=image_geometry(x)
    out=[]
    for nameg,mask in (("digits",y<10),("letters",y>=10)):
        z={"arm":arm,"seed":seed,"dataset":name,"group":nameg,
           "n":int(mask.sum())}
        for field in ("long_side","width","height","mass"):
            zz=np.asarray(geo[field])[mask]
            z[field+"_mean"]=float(zz.mean())
            z[field+"_std"]=float(zz.std())
        out.append(z)
    return out

def binary_size_probe(xtrain,ytrain,xval,yval):
    train=image_geometry(xtrain)["long_side"].reshape(-1,1)
    val=image_geometry(xval)["long_side"].reshape(-1,1)
    model=Pipeline([("std",StandardScaler()),
      ("logistic",LogisticRegression(class_weight="balanced",random_state=42,max_iter=1000))])
    model.fit(train,(ytrain>=10).astype(int))
    q=model.predict_proba(val)[:,1]
    return {"bal_acc":float(balanced_accuracy_score(yval>=10,q>=.5)),
            "auc":float(roc_auc_score(yval>=10,q))}

def prepareraw(official,writers,arm):
    ratio,stroke=ARMS[arm]
    raw=official["X_train"]
    yy=official["y_train"]
    # CoM-only is exactly the EXP-CMB-004 reference condition.
    main=process(raw,"com",20)
    changed={"method":arm,"requested_ratio":ratio,"actual_ratio":0.,
             "n_aug":0,"n_stroke_changed":0,"n_stroke_rejected":0,
             "official_train_rows":len(raw)}
    targets=[]
    if ratio:
        ids,target_map=nested_subset(yy,ratio)  # Normal EXP015 nested low-ratio group
        half,_=balanced_indices(yy,seed=20261116)
        augmented=scale_base(raw,ids,target_map)
        augmented,stroke_stats=stroke_jitter(augmented,stroke,ids,half)
        main[ids]=process(augmented,"com",20)
        changed.update({"n_aug":len(ids),"actual_ratio":len(ids)/len(yy),
             "n_stroke_changed":stroke_stats["modified"],
             "n_stroke_rejected":stroke_stats["rejected"]})
        for cl in CLASSES:
            for length in TARGETS:
                ct=int(np.sum((yy[ids]==cl)&(target_map[ids]==length)))
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":ct})
                assert ct>=1
        for cl in CLASSES:
            cc=[x["train_images_assigned"] for x in targets if x["class"]==LABELS[cl]]
            assert max(cc)-min(cc)<=1
        assert abs(len(ids)/len(yy)-ratio)<.002
    else:
        for cl in CLASSES:
            for length in TARGETS:
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":0})
    transformed_w={w:process(writers[w]["X"],"com",20) for w in WRITERS}
    assert main.shape==(76800,784) and len(transformed_w)==4
    return main,transformed_w,changed,targets

def make_scaled(official,writers):
    ref={}
    for scale in SCALES:
        # exact Normal EXP012 intervention; raw image BBox-centered affine, then CoM
        ref[("official",scale)]=process(fixed_scale(official["X_val"],scale),"com",20)
        for w in WRITERS:
            ref[(w,scale)]=process(fixed_scale(writers[w]["X"],scale),"com",20)
    return ref

def eval_scales(model,raw_map,y,dataset,writer,arm,seed):
    rows=[]; pair_rows=[]; focus=[]; class_errors=[]; family_errors=[]
    probs={}
    for sc in SCALES:
        xx=raw_map[(writer if dataset=="OOF" else "official",sc)]
        q=np.asarray(model.predict_proba(xx),dtype=np.float64)
        assert np.array_equal(model.named_steps["model"].classes_,CLASSES)
        p=q.argmax(axis=1).astype(int)
        letter=q[:,10:].sum(axis=1)
        probs[sc]=(p,letter)
    _,base_letter=probs[1.]
    base_family=base_letter>=.5
    for sc in SCALES:
        p,letter=probs[sc]
        for group,z,cls in (("all",np.ones(len(y),dtype=bool),CLASSES),
             ("digits",y<10,np.arange(10)),("letters",y>=10,np.arange(10,16))):
            m=metrics(y[z],p[z],cls)
            flip=(letter[z]>=.5)!=(base_family[z])
            rows.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
               "scale":sc,"group":group,**m,
               "family_flip":float(flip.mean()),
               "mean_abs_delta_p_letter":float(np.abs(letter[z]-base_letter[z]).mean()),
               "mean_p_letter":float(letter[z].mean())})
        for cls_i in CLASSES:
            z=y==cls_i
            pz=p[z];ll=letter[z];bb=base_letter[z]
            focus.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
                "scale":sc,"actual":LABELS[cls_i],"n":int(z.sum()),
                "correct":int(np.sum(pz==cls_i)),
                "accuracy":float(np.mean(pz==cls_i)),
                "family_flip":float(np.mean((ll>=.5)!=(bb>=.5))),
                "mean_abs_delta_p_letter":float(np.mean(np.abs(ll-bb))),
                "mean_p_letter":float(ll.mean())})
        sc_name=dataset+"_scale_"+str(sc)
        pair_rows+=confusion_pairs(y,p,writer,arm,seed,sc_name)
        class_errors+=total_errors_by_class(y,p,writer,arm,seed,sc_name)
        family_errors+=family_error_breakdown(y,p,writer,arm,seed,sc_name)
    return rows,pair_rows,focus,probs[1.][0],class_errors,family_errors

def output_plots(y,p,out,arm,seed):
    cm=confusion_matrix(y,p,labels=CLASSES)
    for k in ("counts","row_normalized"):
        v=cm if k=="counts" else 100*cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
        fig,ax=plt.subplots(figsize=(10,8))
        im=ax.imshow(v,cmap="viridis",vmin=0,vmax=160 if k=="counts" else 100)
        fig.colorbar(im,ax=ax,fraction=.045,pad=.04)
        ax.set(xticks=np.arange(16),yticks=np.arange(16),
               xticklabels=list(LABELS),yticklabels=list(LABELS),
               xlabel="Predicted",ylabel="True",
               title=f"EXP-CMB-005 {arm} seed={seed} Combined OOF {k}")
        fig.tight_layout()
        fig.savefig(out/f"OOF_confusion_{k}.png",dpi=145)
        plt.close(fig)

def run(arm,seed,out,depth):
    if arm!="scale30_stroke28" or seed not in SEEDS or depth not in (128,256):raise ValueError((arm,seed,depth))
    if out.exists(): raise FileExistsError(out)
    out.mkdir(parents=True)
    (out/"models").mkdir()
    t0=time.perf_counter()
    original=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
    handmade=ROOT/"02_data"/"Hand_made"
    off=load_official(original)
    writers=load_writers(handmade)
    assert off["sha256"]==V3_SHA
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    assert len(off["X_train"])==76800 and len(off["X_val"])==19200
    tpre=time.perf_counter()
    xtrain_off,xwriters,info,targets=prepareraw(off,writers,arm)
    xval=process(off["X_val"],"com",20)
    scale_map=make_scaled(off,writers)
    preprocess_time=time.perf_counter()-tpre
    official_probe=binary_size_probe(xtrain_off,off["y_train"],xval,off["y_val"])
    savecsv(out/"target_assignments.csv",targets)
    features=geom_rows("official_train",xtrain_off,off["y_train"],arm,seed)
    features+=geom_rows("official_val",xval,off["y_val"],arm,seed)
    for w in WRITERS:features+=geom_rows("writer_"+str(w),xwriters[w],writers[w]["y"],arm,seed)
    savecsv(out/"geometry_distributions.csv",features)
    foldrows=[];pairs=[];class_errors=[];family_errors=[];scale_rows=[];class_rows=[];scale_class_rows=[];pred_rows=[];loss=[];fit=[]
    pooled=[]
    for held in WRITERS:
        trwriters=[w for w in WRITERS if w!=held]
        X=np.concatenate([xtrain_off]+[xwriters[w] for w in trwriters])
        y=np.concatenate([off["y_train"]]+[writers[w]["y"] for w in trwriters])
        assert X.shape==(78720,784) and len(y)==78720
        mlp=Pipeline([("scale_0_1",PixelScaler255()),
                      ("model",MLPClassifier(random_state=seed,hidden_layer_sizes=(256,depth)))])
        started=time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            mlp.fit(X,y)
        sec=time.perf_counter()-started
        mlpnet=mlp.named_steps["model"]
        assert mlpnet.n_features_in_==784
        for it,lossval in enumerate(mlpnet.loss_curve_):
            loss.append({"arm":arm,"seed":seed,"heldout":held,
                        "iteration":it+1,"loss":float(lossval)})
        fit.append({"arm":arm,"seed":seed,"heldout":held,
                    "train_n":len(y),"heldout_n":640,
                    "fit_seconds":sec,"iterations":mlpnet.n_iter_,
                    "last_loss":float(mlpnet.loss_),
                    "warnings":" | ".join(str(w.message) for w in caught)})
        fout=out/"models"/f"heldout_{held}.joblib"
        joblib.dump(mlp,fout,compress=3)
        handwritten_y=writers[held]["y"]
        outcome=mlp.predict(xwriters[held]).astype(int)
        pp=mlp.predict(xval).astype(int)
        for ds,ytrue,pred in (("OOF",handwritten_y,outcome),
                              ("OfficialVal",off["y_val"],pp)):
            maskset=(("all",np.ones(len(ytrue),bool),CLASSES),
                     ("digits",ytrue<10,np.arange(10)),
                     ("letters",ytrue>=10,np.arange(10,16)))
            for g,z,classes in maskset:
                foldrows.append({"arm":arm,"seed":seed,"heldout":held,"dataset":ds,
                      "group":g,**metrics(ytrue[z],pred[z],classes)})
        prec,rec,f1,_=precision_recall_fscore_support(handwritten_y,outcome,
              labels=CLASSES,zero_division=0)
        for ci in CLASSES:
            z=handwritten_y==ci
            class_rows.append({"arm":arm,"seed":seed,"writer":held,
               "class":LABELS[ci],"n":int(z.sum()),"correct":int(np.sum(outcome[z]==ci)),
               "precision":float(prec[ci]),"recall":float(rec[ci]),"f1":float(f1[ci])})
        for i in range(640):
            pred_rows.append({"arm":arm,"seed":seed,"writer":held,"index":i,
                "actual":LABELS[int(handwritten_y[i])],
                "pred":LABELS[int(outcome[i])]})
        pooled.append(outcome)
        pairs+=confusion_pairs(handwritten_y,outcome,held,arm,seed,"OOF_original")
        pairs+=confusion_pairs(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        class_errors+=total_errors_by_class(handwritten_y,outcome,held,arm,seed,"OOF_original")
        class_errors+=total_errors_by_class(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        family_errors+=family_error_breakdown(handwritten_y,outcome,held,arm,seed,"OOF_original")
        family_errors+=family_error_breakdown(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        for ds,ytrue,wid in (("OOF",handwritten_y,held),
                             ("OfficialVal",off["y_val"],held)):
            z,p2,c,base,ce,fe=eval_scales(mlp,scale_map,ytrue,ds,wid,arm,seed)
            scale_rows+=z;pairs+=p2;scale_class_rows+=c
            class_errors+=ce;family_errors+=fe
            if ds=="OOF":assert np.array_equal(base,outcome)
            else:assert np.array_equal(base,pp)
        print("FOLD_COMPLETED",arm,seed,held,
              "OOF",round(np.mean(outcome==handwritten_y),6),
              "VAL",round(np.mean(pp==off["y_val"]),6),
              "fit_s",round(sec,2),flush=True)
        del X,y,mlp
    hand_y=np.concatenate([writers[w]["y"] for w in WRITERS])
    pooled_p=np.concatenate(pooled)
    assert len(hand_y)==2560
    output_plots(hand_y,pooled_p,out,arm,seed)
    savecsv(out/"fold_metrics.csv",foldrows)
    savecsv(out/"oof_per_class.csv",class_rows)
    savecsv(out/"per_class_and_scale.csv",scale_class_rows)
    savecsv(out/"pair_confusions.csv",pairs)
    savecsv(out/"per_class_error_rates.csv",class_errors)
    savecsv(out/"family_confusion_rates.csv",family_errors)
    savecsv(out/"scale_sensitivity.csv",scale_rows)
    savecsv(out/"oof_predictions.csv",pred_rows)
    savecsv(out/"training_loss.csv",loss)
    savecsv(out/"training_time.csv",fit)
    overall=metrics(hand_y,pooled_p)
    meta={"experiment":"EXP-CMB-008","stage":"S2_depth","hidden_layer_sizes":[256,depth],"first_layer_width":256,"second_layer_width":depth,"status":"S2 model trained, selection pending",
          "arm":arm,"seed":seed,"handmade_cv":"Leave-One-Writer-Out 4-Fold",
          "official_n":76800,"official_val_n":19200,"training_handmade_each_fold":1920,
          "oof_n":2560,"scale_interventions":SCALES,"class_balanced_targets":TARGETS,
          "official_train_only_augmentation":True,
          "handmade_train_preprocess":"CoM-only",
          "heldout_and_official_val_preprocess":"CoM-only",
          "augmentation_seed":20261116,"augmentation":info,
          "preregistration":"EXP-CMB-008 S2; first hidden width 256; second hidden layer 128 or256, Size30/Stroke28 and CoM frozen; no test",
          "hyperparameter_search":True,"only_changed_parameter":"add_second_hidden_layer",
          "fixed_params":{"activation":"relu","solver":"adam","alpha":0.0001,
             "learning_rate_init":0.001,"max_iter":200,"early_stopping":False},
          "official_sha256":off["sha256"],
          "handmade_sha256":{str(w):writers[w]["sha256"] for w in WRITERS},
          "official_train_size_binary_probe":official_probe,
          "oof_metrics":overall,"fit_total_seconds":sum(r["fit_seconds"] for r in fit),
          "preprocessing_seconds":preprocess_time,
          "wall_seconds":time.perf_counter()-t0,
          "original_test_used":False,"professor_test_used":False,
          "new_handmade_480_used":False,
          "versions":{"numpy":np.__version__,"scikit_learn":sklearn.__version__},
          "model_sha256":{str(w):hashlib.sha256((out/"models"/f"heldout_{w}.joblib").read_bytes()).hexdigest()
                          for w in WRITERS}}
    (out/"manifest.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("COMPLETED",arm,seed,json.dumps(overall),flush=True)

if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--arm",choices=("scale30_stroke28",),default="scale30_stroke28")
    ap.add_argument("--depth",type=int,choices=(128,256),required=True)
    ap.add_argument("--seed",type=int,choices=SEEDS,required=True)
    ap.add_argument("--out",type=Path,required=True)
    ar=ap.parse_args()
    run(ar.arm,ar.seed,ar.out,ar.depth)


## exp_cmb008_s1_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_aggregate.py)  
Git blob: `17381023fe5802b73523b1e299765d78c433dca1`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 S1: compare archived width100 vs new widths on identical writer OOF.
No fitting or Test use. v2: aim four directional error rates <5%, maximize OOF accuracy, guard against overfitting.

CLI 인자 (정적 추출): `--repo`, `--incoming`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s1_aggregate.py
#!/usr/bin/env python3
"""EXP-CMB-008 S1: compare archived width100 vs new widths on identical writer OOF.
No fitting or Test use. v2: aim four directional error rates <5%, maximize OOF accuracy, guard against overfitting.
"""
import argparse,csv,json,statistics
from collections import Counter,defaultdict
from pathlib import Path

SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
WIDTHS=(64,100,128,256)
WATCH=(("0","D"),("8","B"),("D","0"),("B","8"),("5","F"),("7","9"),("7","A"),("4","F"))
BASE=Path("05_model_optimize/MLP/Combined/results/Balanced_Scale_Stroke/stage_low20to30/raw/scale30_stroke28")
def readcsv(p):
    with p.open(encoding="utf-8-sig",newline="") as f:return list(csv.DictReader(f))
def pct(v):return f"{100*v:.3f}%"
def read_arm(repo,incoming,width):
    d={}
    for s in SEEDS:
        p=repo/BASE/("seed_"+str(s)) if width==100 else incoming/("cmb008-w"+str(width)+"-s"+str(s))
        assert p.is_dir(),p
        m=json.loads((p/"manifest.json").read_text())
        assert m["arm"]=="scale30_stroke28" and int(m["seed"])==s
        assert m["augmentation_seed"]==20261116 and m["official_train_only_augmentation"]
        assert m["handmade_train_preprocess"]==m["heldout_and_official_val_preprocess"]=="CoM-only"
        assert not (m["original_test_used"] or m["professor_test_used"] or m["new_handmade_480_used"])
        if width==100:
            assert m["experiment"]=="EXP-CMB-005" and m["no_mlp_hyperparameter_search"]
        else:
            assert m["experiment"]=="EXP-CMB-008" and m["stage"]=="S1_width" and m["width"]==width
            assert m["only_changed_parameter"]=="hidden_layer_sizes"
            assert m["fixed_params"]=={"activation":"relu","solver":"adam","alpha":.0001,"learning_rate_init":.001,"max_iter":200,"early_stopping":False}
        oof=readcsv(p/"oof_predictions.csv")
        assert len(oof)==2560
        keys=[(int(x["writer"]),int(x["index"])) for x in oof]
        assert len(set(keys))==2560 and {w for w,i in keys}==set(WRITERS)
        vals=[r for r in readcsv(p/"fold_metrics.csv") if r["dataset"]=="OfficialVal" and r["group"]=="all"]
        assert len(vals)==4 and all(int(x["n"])==19200 for x in vals)
        scales=[x for x in readcsv(p/"per_class_and_scale.csv") if x["dataset"] in ("OOF","OfficialVal") and float(x["scale"]) in (.8,1.2)]
        assert scales
        times=readcsv(p/"training_time.csv")
        assert len(times)==4
        d[s]={"m":m,"oof":oof,"val":vals,"scales":scales,"times":times}
    return d
def summarize(width,d):
    oof=[r for s in SEEDS for r in d[s]["oof"]]
    correct=lambda z:sum(x["actual"]==x["pred"] for x in z)
    n=len(oof);assert n==7680
    seedacc=[correct(d[s]["oof"])/2560 for s in SEEDS]
    val=statistics.mean(float(v["accuracy"]) for s in SEEDS for v in d[s]["val"])
    counts=Counter(x["actual"] for x in oof)
    assert len(counts)==16 and all(n==480 for n in counts.values())
    pairs={a+"->"+b:sum(x["actual"]==a and x["pred"]==b for x in oof) for a,b in WATCH}
    classerror={c:sum(x["actual"]==c and x["pred"]!=c for x in oof)/counts[c] for c in counts}
    writers={str(w):correct([x for x in oof if int(x["writer"])==w])/1920 for w in WRITERS}
    f=defaultdict(lambda:[0.,0])
    for s in SEEDS:
        for r in d[s]["scales"]:
            k=(r["dataset"],r["actual"],r["scale"])
            nn=int(r["n"]); f[k][0]+=nn*float(r["family_flip"]);f[k][1]+=nn
    assert len(f)==64
    flips={ds:max(v[0]/v[1] for k,v in f.items() if k[0]==ds) for ds in ("OOF","OfficialVal")}
    ts=[x for s in SEEDS for x in d[s]["times"]]
    return {"width":width,"oof":correct(oof)/n,"seed_std":statistics.pstdev(seedacc),
       "val":val,"pairs":pairs,"classerror":classerror,"writers":writers,"flip":flips,
       "mean_n_iter":statistics.mean(int(x["iterations"]) for x in ts),
       "mean_fit_seconds":statistics.mean(float(x["fit_seconds"]) for x in ts),
       "warnings":sum(bool(x["warnings"]) for x in ts)}
TARGET_PAIRS=("D->0","B->8","8->B","0->D")
def target_under_5percent(n):
    # Each actual class contributes 160 images/seed x 3 seeds = 480.
    # Strict <5% means at most 23 (NOT 24) errors per direction.
    return all(n["pairs"][p]<24 for p in TARGET_PAIRS)
def check(b,n):
    # Updated user criterion before comparing S1 results, archived as plan 2-B.
    return {
      "oof_accuracy_at_least_baseline":n["oof"]>=b["oof"]-1e-12,
      "official_val_drop_le_0p5pp":n["val"]>=b["val"]-.005-1e-12,
      "writer_drop_le_1pp":all(n["writers"][k]>=b["writers"][k]-.01-1e-12 for k in b["writers"]),
      "class_0_8_B_D_error_no_increase":all(n["classerror"][k]<=b["classerror"][k]+1e-12 for k in ("0","8","B","D")),
      "max_class_family_flip_increase_le_5pp":all(n["flip"][k]<=b["flip"][k]+.05+1e-12 for k in ("OOF","OfficialVal")),
      "D_to_0_no_increase":n["pairs"]["D->0"]<=b["pairs"]["D->0"],
      "B_to_8_no_increase":n["pairs"]["B->8"]<=b["pairs"]["B->8"],
      "8_to_B_no_increase":n["pairs"]["8->B"]<=b["pairs"]["8->B"],
      "0_to_D_no_increase":n["pairs"]["0->D"]<=b["pairs"]["0->D"]}
def paired(b,n):
    fixed=induced=0
    for s in SEEDS:
        x={(int(r["writer"]),int(r["index"])):r for r in b[s]["oof"]}
        y={(int(r["writer"]),int(r["index"])):r for r in n[s]["oof"]}
        assert x.keys()==y.keys()
        for k in x:
            assert x[k]["actual"]==y[k]["actual"]
            old=x[k]["actual"]==x[k]["pred"]
            new=y[k]["actual"]==y[k]["pred"]
            fixed+=not old and new
            induced+=old and not new
    return fixed,induced
def main():
    a=argparse.ArgumentParser();a.add_argument("--repo",type=Path,default=Path("."))
    a.add_argument("--incoming",type=Path,required=True);a.add_argument("--out",type=Path,required=True)
    args=a.parse_args()
    d={w:read_arm(args.repo,args.incoming,w) for w in WIDTHS}
    for w in WIDTHS:
        for s in SEEDS:
            assert d[w][s]["m"]["official_sha256"]==d[100][s]["m"]["official_sha256"]
            assert d[w][s]["m"]["handmade_sha256"]==d[100][s]["m"]["handmade_sha256"]
    m={w:summarize(w,d[w]) for w in WIDTHS}
    assert abs(m[100]["oof"]-.9071614583333334)<1e-9
    assert abs(m[100]["val"]-.961171875)<1e-9
    checks={w:check(m[100],m[w]) for w in WIDTHS}
    target_meets=[w for w in WIDTHS if target_under_5percent(m[w])]
    primary=[w for w in target_meets if all(checks[w].values())]
    # If 5% target is not achieved, fall back to best OOF among candidates that
    # do not worsen any protected error, Writer, OfficialVal or class stability.
    fallback=[w for w in WIDTHS if all(checks[w].values())]
    pool=primary if primary else fallback
    assert 100 in fallback, "baseline must satisfy all non-regression guards"
    top_acc=max(m[w]["oof"] for w in pool)
    near=[w for w in pool if m[w]["oof"]>=top_acc-.002-1e-12]
    # Differences <=0.2pp: prioritize most serious D->0 confusion, then other
    # three directions, seed robustness, OfficialVal, and smaller width.
    near.sort(key=lambda w:(
        m[w]["pairs"]["D->0"],
        sum(m[w]["pairs"][p] for p in ("B->8","8->B","0->D")),
        -m[w]["oof"],m[w]["seed_std"],-m[w]["val"],w))
    nextwidth=near[0]
    status=("TARGET_MET" if primary else
            ("FALLBACK_IMPROVED" if nextwidth!=100 else "BASELINE_RETAINED"))
    result={"status":status,"target_directed_error_rate_strictly_below":.05,
        "denominator_each_direction":480,"max_errors_per_direction":23,
        "priority_direction":"D->0","v2_protocol_updated_during_S1_before_result_comparison":True,
        "zero_error_required":False,
        "target_met_without_guards":target_meets,
        "target_met_and_safeguards_pass":primary,
        "fallback_safeguards_pass":fallback,
        "selected_width_for_S2":nextwidth,"selected_is_final_model":False,
        "selection_reason":(
           "All four error directions <5%, OOF nondecrease, robustness gates passed"
           if primary else "No acceptable model met all four <5% targets; fallback uses OOF accuracy, "
                         "non-increasing key confusion and class error rates plus robustness guards"),
        "original_test_used":False,"professor_test_used":False,
        "baseline_archived_models":12,"new_fits":36,"metrics":m,"gate":checks,
        "paired_vs_baseline":{w:paired(d[100],d[w]) for w in WIDTHS if w!=100}}
    args.out.mkdir(parents=True,exist_ok=True)
    (args.out/"S1_details.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
    columns=["width","oof_accuracy","oof_error_rate","official_val_accuracy","oof_std",
             "D_to_0","D_to_0_rate","B_to_8","B_to_8_rate",
             "8_to_B","8_to_B_rate","0_to_D","0_to_D_rate",
             "class_0_error","class_8_error","class_B_error","class_D_error",
             "oof_worst_flip","val_worst_flip","mean_iter","fit_seconds","warnings",
             "target_all_4_under_5pct","passed_safeguards","failed_gates"]
    rows=[]
    for w in WIDTHS:
        x=m[w];g=checks[w]
        rows.append(dict(zip(columns,[w,x["oof"],1-x["oof"],x["val"],x["seed_std"],
          x["pairs"]["D->0"],x["pairs"]["D->0"]/480,
          x["pairs"]["B->8"],x["pairs"]["B->8"]/480,
          x["pairs"]["8->B"],x["pairs"]["8->B"]/480,
          x["pairs"]["0->D"],x["pairs"]["0->D"]/480,
          x["classerror"]["0"],x["classerror"]["8"],x["classerror"]["B"],x["classerror"]["D"],
          x["flip"]["OOF"],x["flip"]["OfficialVal"],x["mean_n_iter"],x["mean_fit_seconds"],
          x["warnings"],target_under_5percent(x),all(g.values()),
          ";".join(k for k,v in g.items() if not v)])))
    with (args.out/"S1_width_comparison.csv").open("w",encoding="utf-8-sig",newline="") as f:
        writer=csv.DictWriter(f,fieldnames=columns);writer.writeheader();writer.writerows(rows)
    md=["# EXP-CMB-008 · 02 S1 은닉층 너비 실측","",
        "> **판정:** "+status+" / 신규 학습 36개, 기준선 기존 12개 재사용.",
        "> **수정된 사용자 목표:** D→0, B→8, 8→B, 0→D 각각 실제 클래스 내 오분류율 **5% 미만**, 동시에 높은 OOF 정확도와 과적합 가드레일. 0건 강제 조건은 폐기.","",
        "## 가설과 실험 변화",
        "크기·획·CoM, Data SHA, Seed, writer split과 나머지 HP 고정. 표현 용량의 효과를 보기 위해 너비 64/128/256을 기준 100과 비교했다.",
        "각 오분류율의 분모는 실제 클래스당 480회(160장/Seed × 3Seed). <5% 기준은 23건 이하.","",
        "| Width | OOF Acc. | OOF 오분류율 | OfficialVal | D→0 건수(%) | B→8 건수(%) | 8→B 건수(%) | 0→D 건수(%) | <5% 목표 | 가드레일 위반 |",
        "|---:|---:|---:|---:|---:|---:|---:|---:|---|---|"]
    for row in rows:
        def directed(k):
            return str(row[k])+" ("+pct(row[k+"_rate"])+")"
        md.append("| "+str(row["width"])+" | "+pct(row["oof_accuracy"])+" | "+
             pct(row["oof_error_rate"])+" | "+pct(row["official_val_accuracy"])+" | "+
             directed("D_to_0")+" | "+directed("B_to_8")+" | "+directed("8_to_B")+" | "+
             directed("0_to_D")+" | "+("충족" if row["target_all_4_under_5pct"] else "미달")+
             " | "+(row["failed_gates"] or "없음")+" |")
    md += ["","## 선택 결과와 다음 단계",
        "- **목표 4방향 모두 <5% 도달 후보(가드레일 포함):** "+(str(primary) if primary else "없음"),
        "- **선정한 S2 비교 기준 너비:** "+str(nextwidth),
        "- **결정:** "+result["selection_reason"],
        "- **D→0 우선 관찰:** 기존 71건/480회(14.79%)이며 다른 방향 54건/43건/19건과 함께 악화 여부를 확인했다.",
        "- **과적합 감시:** Writer별 Accuracy, 3Seed 변동, OfficialVal 손실, OOF/Val 16클래스 Family Flip, 반복횟수, Loss와 경고를 개별 JSON/CSV로 보존한다.",
        "- **새 모델 최종 확정 아님:** 후속 S2~S6에서 순차적으로 평가한다. 독립 Test는 아직 사용하지 않는다.",
        "- **이력:** 초기 Zero-error 요구는 S1 학습 개시 후, 결과 비교 전 사용자가 수정했고 이 문서에는 변경된 <5% 기준을 적용했다.",""]
    # Report enough evidence to reconstruct the decision rather than only a winner.
    md+=["", "## 작성자·클래스별 정확도와 과적합 위험",
         "다음 표의 네 숫자는 각 실제 클래스의 **전체 오분류율**이며, 위 표의 특정 방향 오분류율과 구별한다.",
         "| Width | 실제 0 오류 | 실제 8 오류 | 실제 B 오류 | 실제 D 오류 | Writer 최저 OOF | Seed Std | OOF 최대 클래스 Flip | OfficialVal 최대 Flip | 수렴 경고 Fold |",
         "|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for w in WIDTHS:
        x=m[w]
        md.append("| "+str(w)+" | "+" | ".join(pct(x["classerror"][cl]) for cl in ("0","8","B","D"))+
          " | "+pct(min(x["writers"].values()))+" | "+pct(x["seed_std"])+
          " | "+pct(x["flip"]["OOF"])+" | "+pct(x["flip"]["OfficialVal"])+
          " | "+str(x["warnings"])+" |")
    md+=["", "### Writer별 OOF Accuracy (동일 보류 Writer·Seed 집계)",
         "| Width | Writer 41 | Writer 42 | Writer 43 | Writer 44 | 평균 학습 반복 횟수 | 평균 Fold 학습 시간(초) |",
         "|---:|---:|---:|---:|---:|---:|---:|"]
    for w in WIDTHS:
        x=m[w]
        md.append("| "+str(w)+" | "+" | ".join(pct(x["writers"][str(k)]) for k in WRITERS)+
          " | "+f'{x["mean_n_iter"]:.2f}'+" | "+f'{x["mean_fit_seconds"]:.2f}'+" |")
    md+=["", "## 동일 이미지 교정과 신규 오답 (100뉴런 모델 대비)",
         "| 후보 Width | 기존 오답을 정답으로 교정 | 기존 정답을 새로 오분류 | 순개선 |",
         "|---:|---:|---:|---:|"]
    for w in (64,128,256):
        fixed,created=result["paired_vs_baseline"][w]
        md.append("| "+str(w)+" | "+str(fixed)+" | "+str(created)+" | "+
                  ("+" if fixed-created>=0 else "")+str(fixed-created)+" |")
    md+=["", "## 후보 탈락·선정 근거 및 다음 실험에서 검증할 가설",
         "- **64뉴런:** 기준선 대비 OOF 정확도와 OfficialVal이 낮아지고 D→0/B→8 등 오분류가 늘면 탈락한다.",
         "- **128뉴런:** D→0과 8→B가 줄어도 B→8 및 0→D 오류가 늘었다면 특정 클래스 개선을 전체 개선으로 오해하지 않는다.",
         "- **256뉴런:** 정확도·4방향 오분류·Writer별 일반화·0/8/B/D 클래스 오류 가드레일을 통과하면 다음 깊이 실험의 단층 기준으로 삼는다.",
         "- **S2 실험:** 선택한 첫 층 너비 w를 고정하고 (w,), (w, w/2), (w, w) 깊이 후보를 비교한다. 첫 층 동일 조건에서 추가 은닉층 효과를 검증하는 순서다.",
         "- **과적합 주의:** 학습 반복 횟수가 적거나 수렴 경고가 없다는 이유만으로 과적합이 없다고 판단하지 않는다. 반복 사용한 4 Writer의 OOF에서도 탐색 과적합이 가능하며, 별도 보호한 최종 Test 전까지 일반화 성능을 확정하지 않는다.",
         "- **프로토콜 변경 이력:** S1 실행 중 사용자가 기존 0건 규칙을 네 방향 각각 5% 미만 목표로 수정했다. 실측 결과가 확정되기 전 변경한 규칙을 우선 적용하고 과거 규칙도 사전등록 문서에서 확인 가능하다.",
         ""]

    (args.out/"S1_Stage_Results.md").write_text("\n".join(md),encoding="utf-8")
    print("S1_DECISION",json.dumps({"status":result["status"],"provisional":nextwidth,
        "target_pass":primary,"rows":rows},ensure_ascii=False),flush=True)
if __name__=="__main__":main()


## exp_cmb008_s2_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_aggregate.py)  
Git blob: `5a012e52c8d9dfbce178635262c52301f5882f5b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 S2: second-layer depth comparison, no model refit or Test access.

Frozen first hidden width=256 from S1; evaluate (256,), (256,128), (256,256).
Same 30% Size / 28% Stroke + CoM, seeds, writer folds and default other HP.
Prioritized gate: each actual-class directional D->0/B->8/8->B/0->D rate <5%,
otherwise best OOF accuracy without regression in those errors and guardrails.

CLI 인자 (정적 추출): `--new`, `--s1-reference`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s2_aggregate.py
#!/usr/bin/env python3
"""EXP-CMB-008 S2: second-layer depth comparison, no model refit or Test access.

Frozen first hidden width=256 from S1; evaluate (256,), (256,128), (256,256).
Same 30% Size / 28% Stroke + CoM, seeds, writer folds and default other HP.
Prioritized gate: each actual-class directional D->0/B->8/8->B/0->D rate <5%,
otherwise best OOF accuracy without regression in those errors and guardrails.
"""
import argparse,csv,json,sys
from pathlib import Path
from exp_cmb008_s1_aggregate import SEEDS,WRITERS,readcsv,summarize,check,target_under_5percent,paired,pct

ARCHS=("256","256_128","256_256")
WATCH=("D->0","B->8","8->B","0->D")

def artifact(incoming,ref,arch,seed):
    return ((ref/f"cmb008-w256-s{seed}") if arch=="256" else
            (incoming/f"cmb008-s2-d{arch.split('_')[1]}-s{seed}"))

def read_arch(incoming,ref,arch):
    result={}
    for seed in SEEDS:
        p=artifact(incoming,ref,arch,seed)
        assert p.is_dir(),p
        m=json.loads((p/"manifest.json").read_text(encoding="utf-8"))
        assert m["experiment"]=="EXP-CMB-008" and m["arm"]=="scale30_stroke28"
        assert int(m["seed"])==seed and m["augmentation_seed"]==20261116
        assert m["handmade_train_preprocess"]=="CoM-only"
        assert m["heldout_and_official_val_preprocess"]=="CoM-only"
        assert m["official_train_only_augmentation"]
        assert not any(m[k] for k in ("original_test_used","professor_test_used","new_handmade_480_used"))
        if arch=="256":
            assert m["stage"]=="S1_width" and m["width"]==256
            assert m["only_changed_parameter"]=="hidden_layer_sizes"
        else:
            assert m["stage"]=="S2_depth"
            assert m["hidden_layer_sizes"]==[256,int(arch.split("_")[1])]
            assert m["only_changed_parameter"]=="add_second_hidden_layer"
        assert m["fixed_params"]=={
            "activation":"relu","solver":"adam","alpha":0.0001,
            "learning_rate_init":0.001,"max_iter":200,"early_stopping":False}
        oof=readcsv(p/"oof_predictions.csv")
        assert len(oof)==2560 and len({(int(x["writer"]),int(x["index"])) for x in oof})==2560
        vals=[x for x in readcsv(p/"fold_metrics.csv") if x["dataset"]=="OfficialVal" and x["group"]=="all"]
        assert len(vals)==4 and all(int(x["n"])==19200 for x in vals)
        scales=[x for x in readcsv(p/"per_class_and_scale.csv") if
                x["dataset"] in ("OOF","OfficialVal") and float(x["scale"]) in (.8,1.2)]
        assert scales
        times=readcsv(p/"training_time.csv")
        assert len(times)==4
        result[seed]={"m":m,"oof":oof,"val":vals,"scales":scales,"times":times}
    return result

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--new",type=Path,required=True)
    ap.add_argument("--s1-reference",type=Path,required=True)
    ap.add_argument("--out",type=Path,required=True)
    a=ap.parse_args()
    data={arch:read_arch(a.new,a.s1_reference,arch) for arch in ARCHS}
    for arch in ARCHS:
        for seed in SEEDS:
            assert data[arch][seed]["m"]["official_sha256"]==data["256"][seed]["m"]["official_sha256"]
            assert data[arch][seed]["m"]["handmade_sha256"]==data["256"][seed]["m"]["handmade_sha256"]
    measured={arch:summarize(arch,data[arch]) for arch in ARCHS}
    base=measured["256"]
    assert abs(base["oof"]-.9221354166666667)<1e-9,base["oof"]
    assert abs(base["val"]-.9704427083333333)<1e-9,base["val"]
    checks={arch:check(base,measured[arch]) for arch in ARCHS}
    all_under=[arch for arch in ARCHS if target_under_5percent(measured[arch])]
    primary=[arch for arch in all_under if all(checks[arch].values())]
    fallback=[arch for arch in ARCHS if all(checks[arch].values())]
    assert "256" in fallback
    pool=primary or fallback
    bestacc=max(measured[k]["oof"] for k in pool)
    near=[k for k in pool if measured[k]["oof"]>=bestacc-.002-1e-12]
    near.sort(key=lambda k:(measured[k]["pairs"]["D->0"],
            sum(measured[k]["pairs"][q] for q in WATCH if q!="D->0"),
            -measured[k]["oof"],measured[k]["seed_std"],
            -measured[k]["val"],0 if k=="256" else int(k.split("_")[1])))
    winner=near[0]
    status="TARGET_MET" if primary else ("FALLBACK_IMPROVED" if winner!="256" else "BASELINE_RETAINED")
    paired_metrics={arch:paired(data["256"],data[arch]) for arch in ARCHS if arch!="256"}
    report={"experiment":"EXP-CMB-008","stage":"S2_depth","status":status,
        "target_rate_each_direction":"strictly < 5%","denominator_per_actual_class":480,
        "strict_target_max_count":23,"priority_confusion":"D->0",
        "selected_arch_for_S3":winner,"selected_is_final_hp":False,
        "primary_pass":primary,"safe_fallback_candidates":fallback,
        "data_sha_verified_against_s1":True,
        "new_fits":24,"s1_models_reused":12,
        "metrics":measured,"checks":checks,"paired":paired_metrics,
        "original_test_used":False,"professor_test_used":False,
        "selection_reason":"OOF accuracy first under <5% target; if not achievable, no-increase in four directions and protected classes plus Writer/Val/Flip safeguards"}
    a.out.mkdir(parents=True,exist_ok=True)
    (a.out/"S2_details.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    cols=["arch","oof_accuracy","oof_error_rate","official_val_accuracy","seed_std",
          "D_to_0","D_to_0_rate","B_to_8","B_to_8_rate",
          "8_to_B","8_to_B_rate","0_to_D","0_to_D_rate",
          "class_0_error","class_8_error","class_B_error","class_D_error",
          "oof_worst_flip","official_val_worst_flip","mean_n_iter","mean_fit_seconds",
          "warning_folds","target_4_under_5pct","safeguards_pass","failed_gates"]
    rows=[]
    for arch in ARCHS:
        x=measured[arch]
        c=checks[arch]
        vals=[arch,x["oof"],1-x["oof"],x["val"],x["seed_std"]]
        for pair in WATCH:
            vals.extend([x["pairs"][pair],x["pairs"][pair]/480])
        vals.extend(x["classerror"][q] for q in ("0","8","B","D"))
        vals.extend([x["flip"]["OOF"],x["flip"]["OfficialVal"],
                     x["mean_n_iter"],x["mean_fit_seconds"],x["warnings"],
                     target_under_5percent(x),all(c.values()),
                     ";".join(k for k,v in c.items() if not v)])
        assert len(vals)==len(cols),(len(vals),len(cols))
        rows.append(dict(zip(cols,vals)))
    with (a.out/"S2_depth_comparison.csv").open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(rows)
    md=["# EXP-CMB-008 · 03 S2 두 번째 은닉층 검증","",
        "> **상태:** "+status+" / (256,) 기준 모델 12개 재사용, (256,128)과 (256,256) 총 신규 24개 학습.",
        "> 동일 Size30/Stroke28/CoM/alpha/LR/Seed/Writer 분할, 변경 변수는 두 번째 은닉층 추가이다.","",
        "## S1 → S2의 인과 흐름",
        "S1은 100→256 첫 층 너비 확장으로 OOF가 90.716%→92.214% 개선되었지만, D→0은 13.96%로 목표 5%에 미달하였다.",
        "따라서 **첫 은닉층 256은 동결**하고, 추가 층이 더 복잡한 형태 특징을 학습하여 D/0 등 오류를 줄이는지 확인한다. 두 번째 층의 128/256 비교는 층 수 및 전체 파라미터 수 증가를 함께 포함하므로 층 수만의 순수 효과라고 해석하지 않는다.","",
        "## 전체 정확도와 주요 혼동쌍 (각 실제 클래스 480회 예측)",
        "| 구조 | OOF Accuracy | OOF 오분류율 | OfficialVal | D→0 | B→8 | 8→B | 0→D | 4개 <5% | 가드레일 위반 |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---|---|"]
    for row in rows:
        def err(name):return str(row[name])+" ("+pct(row[name+"_rate"])+")"
        md.append("| ("+row["arch"].replace("_",",")+",) | "+
                  pct(row["oof_accuracy"])+" | "+pct(row["oof_error_rate"])+" | "+
                  pct(row["official_val_accuracy"])+" | "+
                  " | ".join(err(t.replace("->","_to_")) for t in WATCH)+" | "+
                  ("충족" if row["target_4_under_5pct"] else "미달")+
                  " | "+(row["failed_gates"] or "없음")+" |")
    md+=["","## 클래스별 오분류율·작성자별 정확도·과적합 징후",
        "| 구조 | 실제 0 오류 | 8 오류 | B 오류 | D 오류 | Writer 최저 | Seed std | OOF 최대 Flip | OfficialVal 최대 Flip | 경고 Fold |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for k in ARCHS:
        x=measured[k]
        md.append("| ("+k.replace("_",",")+",) | "+
             " | ".join(pct(x["classerror"][q]) for q in ("0","8","B","D"))+
             " | "+pct(min(x["writers"].values()))+" | "+pct(x["seed_std"])+
             " | "+pct(x["flip"]["OOF"])+" | "+pct(x["flip"]["OfficialVal"])+
             " | "+str(x["warnings"])+" |")
    md+=["","### 동일 이미지의 교정과 신규 오류 (256 단층 대조)",
        "| 구조 | 기존 오답 교정 | 신규 오답 유발 | 순개선 |",
        "|---|---:|---:|---:|"]
    for k in ("256_128","256_256"):
        fixed,created=paired_metrics[k]
        md.append("| "+k+" | "+str(fixed)+" | "+str(created)+" | "+str(fixed-created)+" |")
    md+=["","## 선택 및 다음 단계",
        "- **4방향 모두 5% 미만 & 안전 Gate 통과:** "+(str(primary) if primary else "없음"),
        "- **S3 L2 규제 검증의 고정 구조:** "+winner,
        "- **결정 원칙:** 1차 5% 목표를 확인하고, 미달이면 기존 256 단층 대비 OOF 정확도·네 방향·4개 취약 클래스·Writer/OfficialVal/Flip이 악화되지 않는 후보만 채택한다.",
        "- **과적합 방지:** 학습 Loss, n_iter, 수렴 경고 및 Writer/Seed 분산을 함께 보되 과적합이 완전히 배제됐다고 단정하지 않는다. OOF 작성자는 탐색 중 반복 사용한 개발 데이터다.",
        "- **후속 S3:** 위 선택 구조를 고정하고 alpha=1e-5,1e-4,1e-3,1e-2의 규제 효과를 비교할 것. 이 단계에서는 아직 alpha를 선택하지 않는다.",
        "- Original Test 및 교수님 Test는 HP 선정에 사용하지 않았다.",""]
    (a.out/"S2_Stage_Results.md").write_text("\n".join(md),encoding="utf-8")
    print("S2_DECISION",json.dumps({"status":status,"selected":winner,"primary":primary,
         "fallback":fallback,"rows":rows},ensure_ascii=False),flush=True)
if __name__=="__main__":main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")